# AI-Driven Healthcare & Insurance Analytics
## IBM SkillsBuild Data Analytics with AI Academic Internship
### Conducted by BharatCares in Association with AICTE

| Field | Details |
|-------|---------|
| **Author** | Pappu Ramesh Baraf |
| **Email** | pappubaraf1@gmail.com |
| **LinkedIn** | https://in.linkedin.com/in/pappubarafaiml |
| **GitHub** | https://github.com/PappuBaraf |
| **College** | Finolex Academy of Management & Technology (FAMT), Ratnagiri |
| **Degree** | B.E. CSE – Artificial Intelligence & Machine Learning (2023–2027) |
| **Dataset** | CA Hospital Dataset – Q1 2025 (Synthetic) |
| **Kaggle** | https://www.kaggle.com/datasets/rajkumarpadmanabhan/ca-hospital-dataset-q1-2025 |

> **Note:** The CA Hospital Dataset Q1 2025 is a **synthetic** dataset generated for educational purposes.
> It does not represent real patient records or any specific healthcare organisation.
> All findings are analytical observations only.


## 1. Setup — Import Libraries

In [ ]:
# Standard library
import warnings
import pathlib
import json

# Data
import numpy as np
import pandas as pd

# Visualisation
import matplotlib
matplotlib.use("Agg")          # headless backend for notebook output
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
)

warnings.filterwarnings("ignore")
print("All libraries imported successfully.")
print(f"pandas   : {pd.__version__}")
print(f"numpy    : {np.__version__}")
print(f"sklearn  : ", end="")
import sklearn; print(sklearn.__version__)
print(f"matplotlib: {matplotlib.__version__}")


## 2. Configure Paths

In [ ]:
BASE   = pathlib.Path(".")
DATA   = BASE / "data"
OUT    = BASE / "outputs"
CHARTS = OUT  / "charts"
CLEAN  = DATA / "cleaned"
PREDS  = OUT  / "predictions"

for d in [OUT, CHARTS, CLEAN, PREDS, OUT / "reports"]:
    d.mkdir(parents=True, exist_ok=True)

print("Project paths ready.")
print(f"Data   : {DATA.resolve()}")
print(f"Outputs: {OUT.resolve()}")


## 3. Phase 1 — Dataset Inspection

Dataset: **CA Hospital Dataset – Q1 2025**
Source: Kaggle — rajkumarpadmanabhan/ca-hospital-dataset-q1-2025


In [ ]:
# Load claims_and_billing.csv
CSV_PATH = DATA / "claims_and_billing.csv"
assert CSV_PATH.exists(), f"File not found: {CSV_PATH}. Place claims_and_billing.csv in data/ folder."

cb = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Shape      : {cb.shape[0]:,} rows x {cb.shape[1]} columns")
print(f"Columns    : {list(cb.columns)}")
print()
cb.head(5)


In [ ]:
# Inspect dtypes and missing values
profile = []
for col in cb.columns:
    profile.append({
        "Column"      : col,
        "Data Type"   : str(cb[col].dtype),
        "Missing"     : int(cb[col].isna().sum()),
        "Missing %"   : round(cb[col].isna().mean() * 100, 2),
        "Unique Values": int(cb[col].nunique()),
    })
profile_df = pd.DataFrame(profile)
profile_df.to_csv(OUT / "dataset_profile.csv", index=False)
print("Dataset Profile:")
print(profile_df.to_string(index=False))


In [ ]:
# Key statistics
total   = len(cb)
paid    = (cb["claim_status"] == "Paid").sum()
denied  = (cb["claim_status"] == "Denied").sum()

print(f"Total Claims   : {total:,}")
print(f"Paid Claims    : {paid:,} ({paid/total*100:.2f}%)")
print(f"Denied Claims  : {denied:,} ({denied/total*100:.2f}%)")
print(f"Total Billed   : ${cb['billed_amount'].sum():,.2f}")
print(f"Total Paid     : ${cb['paid_amount'].sum():,.2f}")
print(f"Revenue Gap    : ${cb['billed_amount'].sum() - cb['paid_amount'].sum():,.2f}")
print(f"Avg Billed     : ${cb['billed_amount'].mean():,.2f}")
print(f"Avg Paid       : ${cb['paid_amount'].mean():,.2f}")
print(f"Providers      : {cb['insurance_provider'].nunique()}")
print(f"Duplicate Rows : {cb.duplicated().sum()}")


## 4. Phase 2 — Table Relationships

In [ ]:
# Identify ID columns and potential foreign keys
id_cols = [c for c in cb.columns if c.lower().endswith("_id") or c.lower() == "id"]
print("Identifier columns:", id_cols)

for col in id_cols:
    uniq_ratio = cb[col].nunique() / len(cb)
    key_type   = "Primary Key" if uniq_ratio > 0.95 else "Foreign Key"
    print(f"  {col:<25} unique={cb[col].nunique():,}  ratio={uniq_ratio:.3f}  -> {key_type}")


## 5. Phase 3 — Data Cleaning

In [ ]:
cb_clean = cb.copy()

# Parse billing date
for fmt in ["%d-%m-%Y %H:%M", "%Y-%m-%d"]:
    parsed = pd.to_datetime(cb_clean["claim_billing_date"], format=fmt, errors="coerce")
    if parsed.notna().mean() >= 0.5:
        cb_clean["claim_billing_date"] = parsed
        print(f"Parsed claim_billing_date with format: {fmt}")
        print(f"  Parse rate: {parsed.notna().mean()*100:.1f}%  |  Nulls remaining: {parsed.isna().sum():,}")
        break

# Remove exact duplicates
before = len(cb_clean)
cb_clean.drop_duplicates(inplace=True)
print(f"Duplicate rows removed: {before - len(cb_clean):,}")
print(f"Final row count        : {len(cb_clean):,}")

# Save cleaned file
cb_clean.to_csv(CLEAN / "claims_and_billing_cleaned.csv", index=False)
print("Saved: data/cleaned/claims_and_billing_cleaned.csv")


In [ ]:
# Missing values summary
miss = cb_clean.isna().sum().to_frame("Missing").assign(
    Missing_Pct=lambda x: (x["Missing"] / len(cb_clean) * 100).round(2)
)
print("Missing Values:")
print(miss.to_string())
print()
print("Note: denial_reason nulls = Paid claims (expected, not a data quality issue).")


## 6. Phase 4 — Feature Engineering

In [ ]:
cb_feat = cb_clean.copy()

# Date features
if pd.api.types.is_datetime64_any_dtype(cb_feat["claim_billing_date"]):
    cb_feat["claim_year"]      = cb_feat["claim_billing_date"].dt.year
    cb_feat["claim_month"]     = cb_feat["claim_billing_date"].dt.month
    cb_feat["claim_day"]       = cb_feat["claim_billing_date"].dt.day
    cb_feat["claim_dayofweek"] = cb_feat["claim_billing_date"].dt.dayofweek
    cb_feat["claim_quarter"]   = cb_feat["claim_billing_date"].dt.quarter
    print("Date features extracted: year, month, day, dayofweek, quarter")

# Financial features
cb_feat["payment_ratio"]  = (cb_feat["paid_amount"] / cb_feat["billed_amount"].replace(0, np.nan)).round(4)
cb_feat["unpaid_amount"]  = (cb_feat["billed_amount"] - cb_feat["paid_amount"]).round(2)
cb_feat["high_cost_flag"] = (cb_feat["billed_amount"] > cb_feat["billed_amount"].quantile(0.75)).astype(int)
print(f"Financial features: payment_ratio, unpaid_amount, high_cost_flag")

# Target-derived flag (EDA only — NOT used in ML)
cb_feat["denial_flag"] = (cb_feat["claim_status"] == "Denied").astype(int)
print("Denial flag (EDA only — excluded from ML to prevent leakage)")

# Patient-level aggregates
pt_agg = cb_feat.groupby("patient_id").agg(
    patient_claim_count  = ("billing_id",  "count"),
    patient_total_billed = ("billed_amount","sum"),
    patient_avg_billed   = ("billed_amount","mean"),
).round(2).reset_index()
cb_feat = cb_feat.merge(pt_agg, on="patient_id", how="left")
print(f"Patient aggregates merged. Final shape: {cb_feat.shape}")

cb_feat.to_csv(CLEAN / "claims_and_billing_features.csv", index=False)
print("Saved: data/cleaned/claims_and_billing_features.csv")


## 7. Phase 5 & 6 — Exploratory Data Analysis (EDA) + Visualisations

In [ ]:
# ── Chart 1: Claim Status Distribution ──
fig, ax = plt.subplots(figsize=(7, 4))
cs = cb_feat["claim_status"].value_counts()
colors = ["#3b82d4", "#dc2626"]
bars = ax.bar(cs.index, cs.values, color=colors, width=0.5, edgecolor="white")
ax.set_title("Claim Status Distribution", fontsize=13, fontweight="bold")
ax.set_xlabel("Claim Status"); ax.set_ylabel("Count")
for bar, val in zip(bars, cs.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f"{val:,}", ha="center", fontsize=10, fontweight="bold")
fig.tight_layout()
fig.savefig(CHARTS / "01_claim_status.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Paid: {cs.get('Paid',0):,} ({cs.get('Paid',0)/len(cb_feat)*100:.1f}%)  |  "
      f"Denied: {cs.get('Denied',0):,} ({cs.get('Denied',0)/len(cb_feat)*100:.1f}%)")


In [ ]:
# ── Chart 2: Claims by Insurance Provider ──
fig, ax = plt.subplots(figsize=(8, 4))
ip = cb_feat["insurance_provider"].value_counts().sort_values()
ip.plot(kind="barh", ax=ax, color="#3b82d4", edgecolor="white")
ax.set_title("Claims by Insurance Provider", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of Claims"); ax.set_ylabel("Provider")
fig.tight_layout()
fig.savefig(CHARTS / "02_insurance_provider.png", dpi=120, bbox_inches="tight")
plt.show()
print("Provider distribution:")
print(cb_feat["insurance_provider"].value_counts().to_string())


In [ ]:
# ── Chart 3: Billed Amount Distribution ──
fig, ax = plt.subplots(figsize=(8, 4))
vals = cb_feat["billed_amount"].dropna()
ax.hist(vals, bins=50, color="#3b82d4", edgecolor="white")
ax.axvline(vals.median(), color="orange", linestyle="--", lw=2,
           label=f"Median ${vals.median():,.0f}")
ax.axvline(vals.mean(), color="red", linestyle="--", lw=2,
           label=f"Mean ${vals.mean():,.0f}")
ax.set_title("Billed Amount Distribution", fontsize=13, fontweight="bold")
ax.set_xlabel("Billed Amount (USD)"); ax.set_ylabel("Frequency")
ax.legend(); fig.tight_layout()
fig.savefig(CHARTS / "03_billed_amount_dist.png", dpi=120, bbox_inches="tight")
plt.show()
print(cb_feat["billed_amount"].describe().round(2).to_string())


In [ ]:
# ── Chart 4: Avg Billed vs Paid by Provider ──
fig, ax = plt.subplots(figsize=(9, 5))
pg = cb_feat.groupby("insurance_provider")[["billed_amount","paid_amount"]].mean().sort_values("billed_amount", ascending=False)
x  = np.arange(len(pg))
w  = 0.35
ax.bar(x - w/2, pg["billed_amount"], w, label="Avg Billed", color="#3b82d4")
ax.bar(x + w/2, pg["paid_amount"],   w, label="Avg Paid",   color="#27ae60")
ax.set_xticks(x); ax.set_xticklabels(pg.index, rotation=25, ha="right")
ax.set_title("Average Billed vs Paid Amount by Insurance Provider", fontsize=12, fontweight="bold")
ax.set_ylabel("Amount (USD)"); ax.legend(); fig.tight_layout()
fig.savefig(CHARTS / "04_billed_vs_paid_by_provider.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── Chart 5: Denial Reasons ──
if "denial_reason" in cb_feat.columns:
    fig, ax = plt.subplots(figsize=(9, 5))
    dr = cb_feat["denial_reason"].value_counts().dropna()
    dr.sort_values().plot(kind="barh", ax=ax, color="#dc2626", edgecolor="white")
    ax.set_title("Claim Denial Reasons", fontsize=13, fontweight="bold")
    ax.set_xlabel("Count"); ax.set_ylabel("Denial Reason")
    fig.tight_layout()
    fig.savefig(CHARTS / "05_denial_reasons.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Total denial reason categories: {len(dr)}")
    print(dr.head(10).to_string())


In [ ]:
# ── Chart 6: Payment Method Pie ──
fig, ax = plt.subplots(figsize=(6, 4))
pm = cb_feat["payment_method"].value_counts()
ax.pie(pm.values, labels=pm.index, autopct="%1.1f%%",
       colors=["#3b82d4","#f39c12"], startangle=90, wedgeprops={"edgecolor":"white"})
ax.set_title("Payment Method Distribution", fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(CHARTS / "06_payment_method.png", dpi=120, bbox_inches="tight")
plt.show()
print(pm.to_string())


In [ ]:
# ── Chart 7: Monthly Trends ──
if pd.api.types.is_datetime64_any_dtype(cb_feat["claim_billing_date"]):
    cb_dated = cb_feat.dropna(subset=["claim_billing_date"]).copy()
    cb_dated["month_period"] = cb_dated["claim_billing_date"].dt.to_period("M").astype(str)
    monthly = cb_dated.groupby("month_period").agg(
        claim_count  = ("billing_id",    "count"),
        total_billed = ("billed_amount", "sum"),
    ).reset_index().sort_values("month_period")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(monthly["month_period"], monthly["claim_count"],  marker="o", color="#3b82d4")
    axes[0].set_title("Monthly Claim Volume"); axes[0].tick_params(axis="x", rotation=45)
    axes[0].set_xlabel("Month"); axes[0].set_ylabel("Claims")
    axes[1].plot(monthly["month_period"], monthly["total_billed"], marker="s", color="#27ae60")
    axes[1].set_title("Monthly Total Billed Amount"); axes[1].tick_params(axis="x", rotation=45)
    axes[1].set_xlabel("Month"); axes[1].set_ylabel("Total Billed (USD)")
    fig.tight_layout()
    fig.savefig(CHARTS / "07_monthly_trends.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Monthly Summary:")
    print(monthly.to_string(index=False))
    print(f"Note: {cb_feat['claim_billing_date'].isna().sum():,} self-pay rows excluded (no billing date).")


In [ ]:
# ── Chart 8: Claim Status % by Provider ──
fig, ax = plt.subplots(figsize=(9, 5))
ct     = cb_feat.groupby(["insurance_provider","claim_status"]).size().unstack(fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind="bar", ax=ax, color=["#dc2626","#3b82d4"], edgecolor="white")
ax.set_title("Claim Status % by Insurance Provider", fontsize=12, fontweight="bold")
ax.set_xlabel("Insurance Provider"); ax.set_ylabel("Percentage (%)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right"); ax.legend()
fig.tight_layout()
fig.savefig(CHARTS / "08_status_by_provider.png", dpi=120, bbox_inches="tight")
plt.show()
print("Denial rates by provider:")
denial_rates = (ct.get("Denied", 0) / ct.sum(axis=1) * 100).round(2)
print(denial_rates.to_string())


In [ ]:
# ── Chart 9: High Cost Flag ──
fig, ax = plt.subplots(figsize=(6, 4))
hc = cb_feat["high_cost_flag"].value_counts().sort_index()
ax.bar(["Standard Cost", "High Cost (>75th pct)"], hc.values, color=["#3b82d4","#dc2626"])
ax.set_title("High-Cost Claim Flag (>75th Percentile)", fontsize=12, fontweight="bold")
ax.set_ylabel("Count")
for i, v in enumerate(hc.values):
    ax.text(i, v + 200, f"{v:,}", ha="center", fontsize=10)
fig.tight_layout()
fig.savefig(CHARTS / "09_high_cost_flag.png", dpi=120, bbox_inches="tight")
plt.show()
q75 = cb_feat["billed_amount"].quantile(0.75)
print(f"75th percentile threshold: ${q75:,.2f}")
print(f"High-cost claims: {hc.get(1,0):,}  |  Standard: {hc.get(0,0):,}")


In [ ]:
# ── Chart 10: Payment Ratio by Claim Status ──
fig, ax = plt.subplots(figsize=(7, 4))
for status, grp in cb_feat.groupby("claim_status"):
    vals = grp["payment_ratio"].dropna()
    ax.hist(vals, bins=30, alpha=0.6, label=status)
ax.set_title("Payment Ratio by Claim Status", fontsize=12, fontweight="bold")
ax.set_xlabel("Payment Ratio (paid / billed)"); ax.set_ylabel("Frequency"); ax.legend()
fig.tight_layout()
fig.savefig(CHARTS / "10_payment_ratio.png", dpi=120, bbox_inches="tight")
plt.show()
print("Payment Ratio statistics by claim status:")
print(cb_feat.groupby("claim_status")["payment_ratio"].describe().round(4).to_string())


## 8. Phase 7 & 8 — ML Problem Selection + Leakage Check

**Target variable:** `claim_status` (Paid / Denied)  
**Problem type:** Binary Classification  
**Primary metric:** ROC-AUC (preferred for imbalanced data)


In [ ]:
TARGET = "claim_status"
print(f"Target variable   : {TARGET}")
print(f"Class distribution :")
print(cb_feat[TARGET].value_counts().to_string())
print(f"Class balance      : {cb_feat[TARGET].value_counts(normalize=True).round(4).to_string()}")

# ── Leakage Analysis ──
print()
print("Feature Selection — Leakage Check:")
leakage_log = []
for col in cb_feat.columns:
    if col == TARGET: continue
    nk = col.lower()
    if any(t in nk for t in ["denial_reason","denial","paid_amount","payment_ratio","unpaid","denial_flag"]):
        risk, used, reason = "HIGH", "No",  "Post-outcome or target-derived"
    elif any(t in nk for t in ["billing_id","claim_id","encounter_id","patient_id"]):
        risk, used, reason = "Medium","No",  "Identifier — no predictive signal"
    elif "date" in nk or "time" in nk:
        risk, used, reason = "Low",  "Extracted", "Raw date replaced by date-part features"
    else:
        risk, used, reason = "None", "Yes", "Safe predictor"
    leakage_log.append({"Feature": col, "Used?": used, "Leakage Risk": risk, "Reason": reason})

feat_sel_df = pd.DataFrame(leakage_log)
feat_sel_df.to_csv(OUT / "ml_feature_selection.csv", index=False)
print(feat_sel_df[["Feature","Used?","Leakage Risk"]].to_string(index=False))


## 9. Phase 9 & 10 — Machine Learning + Evaluation

In [ ]:
# ── Build feature matrix (leakage-safe) ──
cb_ml = cb_clean.copy()

# Re-parse date for feature extraction
for fmt in ["%d-%m-%Y %H:%M", "%Y-%m-%d"]:
    parsed = pd.to_datetime(cb_ml["claim_billing_date"], format=fmt, errors="coerce")
    if parsed.notna().mean() >= 0.5:
        cb_ml["feat_claim_year"]      = parsed.dt.year
        cb_ml["feat_claim_month"]     = parsed.dt.month
        cb_ml["feat_claim_dayofweek"] = parsed.dt.dayofweek
        cb_ml["feat_claim_quarter"]   = parsed.dt.quarter
        break

FEATURES = [
    "insurance_provider",
    "payment_method",
    "billed_amount",
    "feat_claim_year",
    "feat_claim_month",
    "feat_claim_dayofweek",
    "feat_claim_quarter",
]
FEATURES = [f for f in FEATURES if f in cb_ml.columns]

print("Features used in ML:")
for f in FEATURES: print(f"  - {f}")
print(f"\nExcluded (leakage/ID): denial_reason, paid_amount, billing_id, claim_id, patient_id, encounter_id")


In [ ]:
# ── Encode target ──
le = LabelEncoder()
y  = le.fit_transform(cb_ml[TARGET].astype(str))
X  = cb_ml[FEATURES].copy()

print(f"Class encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"X shape: {X.shape}  |  y distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

# ── Train/Test Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")


In [ ]:
# ── Build preprocessing + model pipelines ──
numeric_feats  = X.select_dtypes(include=np.number).columns.tolist()
categ_feats    = X.select_dtypes(include="object").columns.tolist()

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
pre = ColumnTransformer([
    ("num", num_pipe, numeric_feats),
    ("cat", cat_pipe, categ_feats),
])

# Variant A — default weights (primary)
pipe_def = Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=1000, random_state=42))])

# Variant B — balanced weights (sensitivity)
pipe_bal = Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])

pipe_def.fit(X_train, y_train)
pipe_bal.fit(X_train, y_train)

pred_def = pipe_def.predict(X_test)
prob_def = pipe_def.predict_proba(X_test)[:, 1]

pred_bal = pipe_bal.predict(X_test)
prob_bal = pipe_bal.predict_proba(X_test)[:, 1]

print("Both model variants trained successfully.")


In [ ]:
# ── Evaluation metrics ──
def evaluate(y_true, y_pred, y_prob, label):
    acc  = round(accuracy_score(y_true, y_pred), 4)
    prec = round(precision_score(y_true, y_pred, zero_division=0), 4)
    rec  = round(recall_score(y_true, y_pred, zero_division=0), 4)
    f1   = round(f1_score(y_true, y_pred, zero_division=0), 4)
    roc  = round(roc_auc_score(y_true, y_prob), 4)
    print(f"  {label}")
    print(f"    Accuracy  : {acc}")
    print(f"    Precision : {prec}")
    print(f"    Recall    : {rec}")
    print(f"    F1-Score  : {f1}")
    print(f"    ROC-AUC   : {roc}  <-- primary metric for imbalanced data")
    return acc, prec, rec, f1, roc

print("=== Model Evaluation Results ===")
print("Variant A — Default Weights:")
a_acc, a_prec, a_rec, a_f1, a_roc = evaluate(y_test, pred_def, prob_def, "Default")
print()
print("Variant B — Balanced Weights:")
b_acc, b_prec, b_rec, b_f1, b_roc = evaluate(y_test, pred_bal, prob_bal, "Balanced")


In [ ]:
# ── Confusion matrices ──
cm_def = confusion_matrix(y_test, pred_def)
cm_bal = confusion_matrix(y_test, pred_bal)
classes = le.classes_

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm_data, title_suf in zip(axes, [cm_def, cm_bal], ["Default Weights","Balanced Weights"]):
    im = ax.imshow(cm_data, cmap="Blues")
    ax.set_title(f"Confusion Matrix\n({title_suf})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(classes); ax.set_yticklabels(classes)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm_data[i, j], ha="center", va="center",
                    color="white" if cm_data[i, j] > cm_data.max() / 2 else "black", fontsize=13)
    plt.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(OUT / "confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

print("Confusion Matrix — Default Weights:")
print(pd.DataFrame(cm_def, index=["Actual: Denied","Actual: Paid"],
                   columns=["Pred: Denied","Pred: Paid"]).to_string())


In [ ]:
# ── Classification report ──
print("Classification Report — Default Weights (Primary Model):")
print(classification_report(y_test, pred_def, target_names=le.classes_, zero_division=0))

cr = classification_report(y_test, pred_def, target_names=le.classes_, output_dict=True, zero_division=0)
cr_df = pd.DataFrame(cr).T.reset_index().rename(columns={"index": "Class"})
cr_df.to_csv(OUT / "classification_report.csv", index=False)


In [ ]:
# ── Save model metrics ──
metrics_df = pd.DataFrame([
    {
        "Model"       : "Logistic Regression (default weights)",
        "Target"      : TARGET,
        "Classes"     : str(list(le.classes_)),
        "Train Size"  : len(X_train),
        "Test Size"   : len(X_test),
        "Accuracy"    : a_acc,
        "Precision"   : a_prec,
        "Recall"      : a_rec,
        "F1-Score"    : a_f1,
        "ROC-AUC"     : a_roc,
        "Class Weight": "default",
        "Note"        : "Primary model — optimises overall accuracy",
    },
    {
        "Model"       : "Logistic Regression (balanced weights)",
        "Target"      : TARGET,
        "Classes"     : str(list(le.classes_)),
        "Train Size"  : len(X_train),
        "Test Size"   : len(X_test),
        "Accuracy"    : b_acc,
        "Precision"   : b_prec,
        "Recall"      : b_rec,
        "F1-Score"    : b_f1,
        "ROC-AUC"     : b_roc,
        "Class Weight": "balanced",
        "Note"        : "Sensitivity model — higher recall for minority (Denied) class",
    },
])
metrics_df.to_csv(OUT / "model_metrics.csv", index=False)
print("Saved: outputs/model_metrics.csv")
print(metrics_df[["Model","Accuracy","Precision","Recall","F1-Score","ROC-AUC"]].to_string(index=False))


In [ ]:
# ── Save predictions ──
results = X_test.copy()
results["actual_label"]           = le.inverse_transform(y_test)
results["predicted_label"]        = le.inverse_transform(pred_def)
results["prediction_probability"] = prob_def.round(4)
results["denial_risk_score"]      = (1 - prob_def).round(4)
results["risk_level"] = pd.cut(
    results["denial_risk_score"],
    bins=[-0.001, 0.10, 0.30, 1.001],
    labels=["Low Risk", "Medium Risk", "High Risk"],
)
results.to_csv(PREDS / "model_predictions.csv", index=False)
print(f"Saved: outputs/predictions/model_predictions.csv  ({len(results):,} rows)")
print("\nRisk level distribution:")
print(results["risk_level"].value_counts().to_string())


In [ ]:
# ── Save model summary JSON ──
model_summary = {
    "target"                  : TARGET,
    "positive_class"          : "Paid",
    "negative_class"          : "Denied",
    "features_used"           : FEATURES,
    "n_train"                 : int(len(X_train)),
    "n_test"                  : int(len(X_test)),
    "accuracy"                : a_acc,
    "precision"               : a_prec,
    "recall"                  : a_rec,
    "f1"                      : a_f1,
    "roc_auc"                 : a_roc,
    "balanced_recall_denied"  : round(recall_score(y_test, pred_bal, zero_division=0, pos_label=0), 4),
    "balanced_roc_auc"        : b_roc,
    "confusion_matrix_default": cm_def.tolist(),
    "confusion_matrix_balanced": cm_bal.tolist(),
    "class_encoding"          : {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))},
    "model_note"              : (
        "Two Logistic Regression variants trained. Default weights = primary model. "
        "ROC-AUC ~0.576 reflects limited discriminative power with 7 pre-outcome features "
        "in synthetic data. denial_reason excluded to prevent target leakage."
    ),
    "class_imbalance_note"    : (
        "91.4% Paid / 8.6% Denied. ROC-AUC is the meaningful metric — "
        "not raw accuracy."
    ),
}
with open(OUT / "model_summary.json", "w") as fh:
    json.dump(model_summary, fh, indent=2)
print("Saved: outputs/model_summary.json")


## 10. Phase 12 — AI-Driven Insights

In [ ]:
denial_rate = round(denied / total * 100, 2)
avg_b = round(cb_feat["billed_amount"].mean(), 2)
avg_p = round(cb_feat["paid_amount"].mean(), 2)

insight_rows = [
    {"Category": "Observations", "Insight": f"The dataset contains {total:,} claim/billing records covering Q1 2025."},
    {"Category": "Observations", "Insight": f"The denial rate is {denial_rate}% ({denied:,} of {total:,} claims were denied)."},
    {"Category": "Observations", "Insight": f"The average billed amount is ${avg_b:,.2f}; average paid is ${avg_p:,.2f}."},
    {"Category": "Observations", "Insight": f"Total billed: ${cb_feat['billed_amount'].sum():,.2f}; total paid: ${cb_feat['paid_amount'].sum():,.2f}."},
    {"Category": "Observations", "Insight": f"There are {cb_feat['insurance_provider'].nunique()} insurance providers in the dataset."},
    {"Category": "Possible Insights", "Insight": "Denial rate ~8.6% — reducing duplicate claim submissions is a quick-win opportunity."},
    {"Category": "Possible Insights", "Insight": f"Gap between avg billed (${avg_b:,.2f}) and paid (${avg_p:,.2f}) represents potential revenue leakage."},
    {"Category": "Possible Insights", "Insight": "Duplicate submissions and prior auth are top denial reasons — process standardisation may help."},
    {"Category": "Limitations", "Insight": "Synthetic dataset — patterns do not reflect any real healthcare network."},
    {"Category": "Limitations", "Insight": "Only claims_and_billing.csv available — no patient/clinical context."},
    {"Category": "Operational Actions", "Insight": "Implement real-time duplicate claim detection before submission."},
    {"Category": "Operational Actions", "Insight": "Strengthen prior authorization workflows."},
    {"Category": "Operational Actions", "Insight": "Monitor monthly denial rates by provider to detect trends."},
]

insights_df = pd.DataFrame(insight_rows)
insights_df.to_csv(OUT / "ai_insights.csv", index=False)
print("AI Insights:")
for _, row in insights_df.iterrows():
    print(f"  [{row['Category']}] {row['Insight']}")


## 11. Summary — All Outputs Generated

In [ ]:
import os

print("=== PROJECT OUTPUT SUMMARY ===")
print()
output_files = list(OUT.glob("*.csv")) + list(OUT.glob("*.json")) + list(OUT.glob("*.png"))
chart_files  = list(CHARTS.glob("*.png"))
pred_files   = list(PREDS.glob("*.csv"))

print(f"Output CSVs/JSON : {len(output_files)} files")
print(f"EDA Charts       : {len(chart_files)} PNG files")
print(f"Prediction files : {len(pred_files)} files")
print()
print("Key Results:")
print(f"  Dataset        : {total:,} claims")
print(f"  Denial Rate    : {denial_rate}%")
print(f"  Total Billed   : ${cb_feat['billed_amount'].sum():,.2f}")
print(f"  Revenue Gap    : ${cb_feat['billed_amount'].sum() - cb_feat['paid_amount'].sum():,.2f}")
print(f"  ML Accuracy    : {a_acc}")
print(f"  ML ROC-AUC     : {a_roc}  (primary metric)")
print()
print("To launch the Streamlit dashboard:")
print("  streamlit run app.py")


## Ethical Considerations

- This is a **synthetic dataset** — no real patient data was used.
- This notebook does **not diagnose patients** or provide medical treatment recommendations.
- All findings are analytical observations only — **not causal conclusions**.
- The ML model is for educational demonstration only and is **not production-ready**.
- In a real setting, a denial prediction model would require demographic fairness audits.

---

**Author:** Pappu Ramesh Baraf  
**Email:** pappubaraf1@gmail.com  
**LinkedIn:** https://in.linkedin.com/in/pappubarafaiml  
**GitHub:** https://github.com/PappuBaraf  
**Internship:** IBM SkillsBuild Data Analytics with AI — BharatCares / AICTE
